In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

In [8]:
# Cargar el dataset original
df = pd.read_csv('../data_exam/cars.csv')
df.head()

,Price,Year,Kms,Miles,Fuel,Transmission,Owner,Seller,Drivetrain,Length,Width,Height,Seating
0,5656.0,2017,87150,54152.48265,Petrol,Manual,First,Corporate,FWD,3990,1680,1505,5
1,5040.0,2014,75000,46602.82500,Diesel,Manual,Second,Individual,FWD,3995,1695,1555,5
2,2464.0,2011,67000,41631.85700,Petrol,Manual,First,Individual,FWD,3585,1595,1550,5
3,8948.8,2019,37500,23301.41250,Petrol,Manual,First,Individual,FWD,3995,1745,1510,5
4,21840.0,2018,69000,42874.59900,Diesel,Manual,First,Individual,RWD,4735,1830,1795,7


In [9]:
df.info()
print("\nEstadísticas descriptivas:")
print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 1703 entries, 0 to 1702
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Price         1703 non-null   float64
 1   Year          1703 non-null   int64  
 2   Kms           1703 non-null   int64  
 3   Miles         1703 non-null   float64
 4   Fuel          1703 non-null   str    
 5   Transmission  1703 non-null   str    
 6   Owner         1703 non-null   str    
 7   Seller        1701 non-null   str    
 8   Drivetrain    1703 non-null   str    
 9   Length        1703 non-null   int64  
 10  Width         1703 non-null   int64  
 11  Height        1703 non-null   int64  
 12  Seating       1703 non-null   int64  
dtypes: float64(2), int64(6), str(5)
memory usage: 173.1 KB

Estadísticas descriptivas:
              Price         Year            Kms          Miles       Length  \
count   1703.000000  1703.000000    1703.000000    1703.000000  1703.000000   
mean   13031.

In [10]:
# 2.1 Detección y eliminación de duplicados reales
print(f"Duplicados detectados: {df.duplicated().sum()}")
df = df.drop_duplicates()

# 2.2 Eliminación de la columna 'Miles' por multicolinealidad perfecta con 'Kms'
if 'Miles' in df.columns:
    df = df.drop(columns=['Miles'])

Duplicados detectados: 4


In [11]:
# Corrección crítica: Usamos 'Price' como target real y eliminamos cualquier manipulación artificial
X = df.drop(['Price'], axis=1)
y = df['Price']

# Conversión explícita de tipos de datos object a category para modelos nativos
for col in X.select_dtypes(include='object').columns:
    X[col] = X[col].astype('category')

In [12]:
# División del dataset preservando un conjunto de test independiente
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")

Dimensiones de X_train: (1359, 11)
Dimensiones de X_test: (340, 11)


In [13]:
# Identificar columnas por tipo
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Transformadores individuales
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Procesador global mediante ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

Modelado

In [14]:
# Modelo Base (Dummy Regressor) envuelto en Pipeline
dummy_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])

dummy_cv = cross_validate(dummy_pipe, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"Baseline (Dummy) Validation MAE: {-dummy_cv['test_score'].mean().round(2)}")

Baseline (Dummy) Validation MAE: 8405.73


In [15]:
# Regresión Lineal envuelta en Pipeline
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_cv = cross_validate(lr_pipe, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"Linear Regression Validation MAE: {-lr_cv['test_score'].mean().round(2)}")

Linear Regression Validation MAE: 4053.41


In [16]:
# Random Forest envuelto en Pipeline
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_cv = cross_validate(rf_pipe, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"Random Forest Validation MAE: {-rf_cv['test_score'].mean().round(2)}")

Random Forest Validation MAE: 1999.27


In [18]:
# 1. Identificar de forma explícita qué columnas de X_train son categóricas
# Esto genera una lista de True/False para cada columna, eliminando la fragilidad de 'from_dtype'
is_categorical = [col in X_train.select_dtypes(include=['object', 'category']).columns for col in X_train.columns]

# 2. Instanciar el modelo pasando la máscara booleana directamente
gb_model = HistGradientBoostingRegressor(categorical_features=is_categorical, random_state=42)

# 3. Espacio de búsqueda de hiperparámetros
param_distributions = {
    'max_iter': [50, 100, 150, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, None],
    'min_samples_leaf': [10, 20, 30]
}

# 4. Configurar el RandomizedSearchCV
gb_rs = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

# 5. Ajuste con los datos de entrenamiento
gb_rs.fit(X_train, y_train)

# 6. Mostrar resultados limpios
print(f"Mejores parámetros para HistGradientBoosting: {gb_rs.best_params_}")
print(f"HistGradientBoosting Tuned Validation MAE: {-gb_rs.best_score_:.2f}")

Mejores parámetros para HistGradientBoosting: {'min_samples_leaf': 20, 'max_iter': 100, 'max_depth': None, 'learning_rate': 0.1}
HistGradientBoosting Tuned Validation MAE: 1998.71


Evaluacion del conjunto de test

In [19]:
# Utilizar el mejor modelo para predecir sobre el Hold-out set (X_test)
best_model = gb_rs.best_estimator_
predicciones_test = best_model.predict(X_test)

# Calcular el MAE real definitivo del prototipo
mae_final = mean_absolute_error(y_test, predicciones_test)
print(f"El MAE final real medido en el conjunto de prueba (Test) es: {mae_final:.2f}")

# Guardar los resultados en el DataFrame de prueba para exportación
df_testing_results = X_test.copy()
df_testing_results['Real_Price'] = y_test
df_testing_results['Predicted_Price'] = predicciones_test
df_testing_results.to_csv('predicciones_tasacion.csv', index=False)
print("Archivo 'predicciones_tasacion.csv' exportado correctamente.")

El MAE final real medido en el conjunto de prueba (Test) es: 1808.90
Archivo 'predicciones_tasacion.csv' exportado correctamente.


5. Conclusiones (1 punto)

- Con qué modelo te quedarías para poner en producción? En caso de que no haya ninguno explica porqué.
- Si tuvieras que mejorar el rendimiento del modelo, cuáles son los siguientes pasos que seguirías?

Conclusión Escrita para defender el examen:
1. Elección del Modelo: Seleccionamos el HistGradientBoostingRegressor optimizado dado que obtiene el menor MAE
   en validación cruzada y mantiene una generalización sólida en el set de prueba independiente (evitando el overfitting).

2. Siguientes pasos para mejora: Realizar ingeniería de características avanzada (ej. ratio de kilómetros por año),
   recolectar variables críticas faltantes (marca, modelo, estado del motor) y probar arquitecturas como XGBoost o LightGBM.